# Kelvin-Helmholtz Instability (2D)

This notebook runs the Kelvin-Helmholtz instability benchmark: two counter-streaming layers of different density (`rho1`/`rho2` at `v1`/`v2`), seeded with a single-mode transverse perturbation (`freq`, `w0`, `sigma`) so the roll-up is deterministic and comparable across resolutions rather than seeded from noise. `kelvinHelmholtzCase.configureScheme` also re-points the domain at `[0, L]^2` (a corner-origin box, unlike the symmetric box `buildDomainDescription` returns by default) before deferring to the shared `configureCompressible` setup.

Like `08`–`11`, this is a `particlePlot` (2D field view) case: plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly on `KELVIN_HELMHOLTZ_FIELDS` (exported from `warpSPH.cases.kelvinHelmholtz`) rather than going through `kelvinHelmholtzCase.setupPlot`/`updatePlot`, which do not live-update reliably inside a Jupyter cell in this environment. The density panel uses a diverging colour map (`RdBu`) rather than the uniform one most other cases use, since `rho1`/`rho2` straddle a genuine midpoint here.

`kelvinHelmholtzCase` has no `timestep` hook and leaves `dt` unset in its defaults, so `sampleKHH` picks the CFL-derived `dt` during IC construction and the loop below is a fixed `range(nSteps)`.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/12-Kelvin_Helmholtz.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.kelvinHelmholtz import kelvinHelmholtzCase, KELVIN_HELMHOLTZ_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import math
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `12-kelvin-helmholtz.py`, made explicit and editable here.
# `kelvinHelmholtzCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=kelvinHelmholtzCase.name, scheme=kelvinHelmholtzCase.scheme,
                params=dict(kelvinHelmholtzCase.params)) \
    .merged(**kelvinHelmholtzCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=256,
    dim=2,
    L=1.0,

    # --- time stepping ---------------------------------------------------
    tLimit=4.0,
    # No `dt` here -- `sampleKHH` leaves the CFL-derived value from
    # `computeTimestep` in `config.dt`, and there is no `timestep` hook to
    # re-pick it, so it stays fixed for the whole run.

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- Kelvin-Helmholtz's own knobs (layers + perturbation) -----------------
    params=dict(
        rho1=1.0, rho2=2.0, v1=0.5, v2=-0.5,
        delta=0.025, freq=4.0, w0=0.1, sigma=0.05 / math.sqrt(2.0),
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`kelvinHelmholtzCase.buildSystem` -> `sampleKHH`), not re-derived here.
# `configureScheme` is the case's own -- it re-points the domain at [0, L]^2
# before deferring to the shared compressible setup, see the intro cell.
ctx = buildContext(kelvinHelmholtzCase, spec)
kelvinHelmholtzCase.configureScheme(ctx)
system = kelvinHelmholtzCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(KELVIN_HELMHOLTZ_FIELDS), not
# kelvinHelmholtzCase.setupPlot -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, KELVIN_HELMHOLTZ_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = kelvinHelmholtzCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=kelvinHelmholtzCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = kelvinHelmholtzCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, KELVIN_HELMHOLTZ_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=kelvinHelmholtzCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
